# Compile once, evaluate many states

This walkthrough uses one fixed question: **is the person's age at least 18?**
The question stays the same while the input state changes.

The LLM generates a Python program during `compile()`. Later, `system_one()` runs that program locally and wraps its answer in the TypeSafe SDK response types.


In [1]:
from openai import OpenAI

from jev_heuristic_adapter import HeuristicAdapterClient, Noul
from jev_heuristic_adapter.providers.openai import OpenAIProvider

## 1. Define the question and optional examples

`Noul` represents a yes/no question. `adult` is the name used to retrieve this question's program and answer.

Each example contains a `state` and named `answers`. You may supply zero, one, or many examples; they guide code generation and are not executed as acceptance tests for the generated program.


In [2]:
questions = {"adult": Noul(instructions="Adult if age is over or equal to 18.")}
examples = [
    {"state": {"age": 17}, "answers": {"adult": False}},
    {"state": {"age": 18}, "answers": {"adult": True}},
]

## 2. Choose the model that writes the program

`OpenAIProvider` holds the generation settings. The adapter's core uses the provider interface, while OpenAI-specific calls stay inside the provider.

This example uses `gpt-5.6-luna` with `high` reasoning effort.


In [3]:
provider = OpenAIProvider(OpenAI(), model="gpt-5.6-luna", reasoning_effort="high")
client = HeuristicAdapterClient(provider)

## 3. Compile the question

`compile()` returns a dictionary of compilation artifacts keyed by question name. It generates a program on a cache miss and loads a matching cached program otherwise.

Pass `force=True` when you explicitly want a fresh generation. Syntax and function-interface checks still apply.


In [4]:
programs = client.compile(questions, examples)

## 4. Read the generated Python

`programs["adult"].source` is the source returned by the model. Its entry point is `predict(state)`, which returns `{"answer": value}`. The adapter assigns the question name separately.


In [5]:
print(programs["adult"].source)

def predict(state) -> dict:
    return {"answer": state["age"] >= 18}



## 5. Evaluate new states

The same loaded program handles every age below. These calls do not ask the provider to generate new answers.

For this discrete `Noul` task, the program returns a boolean; the SDK exposes it as `0.0` or `1.0` through `response.nouls["adult"].noul`.


In [6]:
for age in (12, 18, 40):
    response = client.system_one({"age": age}, questions)
    print(age, response.nouls["adult"].noul)

12 0.0
18 1.0
40 1.0


## 6. Load the cached program in a new adapter

A new adapter still calls `compile()` to prepare its in-memory bindings. When the complete compilation messages and provider settings match, it loads the persisted program from the operating system's per-user application data directory.

The same artifact can evaluate another state without an API call. Changes to the question definition, examples, prompt/schema, or provider settings produce a different cache key.


In [7]:
fresh = HeuristicAdapterClient(provider)
cached_programs = fresh.compile(questions, examples)
print(
    "Same artifact:",
    cached_programs["adult"].artifact_id == programs["adult"].artifact_id,
)
print("age=21 ->", fresh.system_one({"age": 21}, questions).nouls["adult"].noul)

Same artifact: True
age=21 -> 1.0
